In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import librosa
import librosa.display
import soundfile as sf
import speech_recognition as sr

from jiwer import wer, cer
from IPython.display import Audio

import whisper

import csv
import os
import tempfile
import wave

from gtts import gTTS


In [ ]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

In [ ]:
audio_signal, sample_rate = librosa.load('speech_01.wav', sr=None)

In [ ]:
sample_rate


In [ ]:
audio_signal

In [ ]:
plt.figure(figsize=(12, 4))
librosa.display.waveshow(audio_signal, sr=sample_rate)
plt.title('Waveform')
plt.xlabel('Time (s)')
plt.ylabel('Amplitude')
plt.show()

# Play the audio in the notebook
Audio('speech_01.wav')

In [ ]:
recognizer = sr.Recognizer()

In [ ]:
file_path = 'speech_01.wav'

In [ ]:
def transcribe_audio(file_path):
    with sr.AudioFile(file_path) as source:
        audio_data = recognizer.record(source)
        text = recognizer.recognize_google(audio_data)
        print(text)
        return text    
transcribed_text = transcribe_audio(file_path)

In [ ]:
ground_truth = """My name is Ivan and I am excited to have you as part of our learning community! 
Before we get started, I’d like to tell you a little bit about myself. I’m a sound engineer turned data scientist,
curious about machine learning and Artificial Intelligence. My professional background is primarily in media production,
with a focus on audio, IT, and communications"""

In [ ]:
calculated_wer = wer(ground_truth, transcribed_text)
calculated_cer = cer(ground_truth, transcribed_text)
print(f"Word Error Rate (WER): {calculated_wer:.4f}")
print(f"Character Error Rate (CER): {calculated_cer:.4f}")

In [ ]:
plt.figure(figsize=(12, 4))
librosa.display.waveshow(audio_signal, sr=sample_rate)
plt.title('Waveform')
plt.xlabel('Time (s)')
plt.ylabel('Amplitude')
plt.show()

# Play the audio in the notebook
Audio('speech_01.wav')

In [ ]:
S = librosa.stft(audio_signal)

In [ ]:
S_db = librosa.amplitude_to_db(abs(S), ref=np.max)

In [ ]:
np.max(S_db)

In [ ]:
plt.figure(figsize=(12, 4))
librosa.display.specshow(data = S_db, sr=sample_rate, x_axis='time', y_axis='log')
plt.colorbar(format='%+2.0f dB')
plt.title('Spectrogram')
plt.xlabel('Time')
plt.ylabel('Frequency')
plt.show()

In [ ]:
signal_filtered = librosa.effects.preemphasis(audio_signal, coef=0.97)
sf.write('filtered_speech_01.wav', signal_filtered, sample_rate)
output_file='filtered_speech_01.wav'

In [ ]:
Audio(output_file)

In [ ]:
transcribed_text = transcribe_audio(output_file)

In [ ]:
model = whisper.load_model("base")

In [ ]:
result = model.transcribe(file_path)

In [ ]:
trasncribed_text_whisper = result["text"]
trasncribed_text_whisper

In [ ]:
result["language"]

In [ ]:
calculated_wer = wer(ground_truth, trasncribed_text_whisper)
calculated_cer = cer(ground_truth, trasncribed_text_whisper)
print(f"Word Error Rate (WER): {calculated_wer:.4f}")
print(f"Character Error Rate (CER): {calculated_cer:.4f}")

In [ ]:
directory_path = './recordings'

In [ ]:
def transcribe_directory_whisper(directory_path):
    transcriptions = []
    for file_name in os.listdir(directory_path):
        if file_name.endswith(".wav"):
            file_path = os.path.join(directory_path, file_name)
            result = model.transcribe(file_path)
            transcription = result["text"]
            transcriptions.append({"file_name": file_name, "transcription": transcription })
    return transcriptions

In [ ]:
transcriptions = transcribe_directory_whisper(directory_path)

In [ ]:
transcriptions


In [ ]:
output_file = "transcriptions.csv"

with open(output_file, mode="w", newline="") as file:
    writer = csv.writer(file)
    writer.writerow(["Track Number", "File Name", "Transcription"])
    for number, transcription in enumerate(transcriptions, start=1):
        writer.writerow([number, transcription['file_name'], transcription['transcription']])

In [ ]:
text = """This is wiredots testing the text to speech recognition, This is just a test by the way. """

In [ ]:
tts = gTTS(text=text, lang='en')

tts.save("./output.mp3")

Audio('output.mp3')